# FCC E-Rate: free GPU census
Additive gpt-oss 20B analysis of all 3,230 readable filings. Outputs are unvalidated machine interpretations. Original pipelines remain intact. Long appendices retain explicit prioritized-window scope.


In [ ]:
# Free T4 x2 only. Original research pipelines and source files remain unchanged.
import os, subprocess, sys, json, time, shutil
from pathlib import Path
repo=Path('/kaggle/temp/e-rate')
repo.parent.mkdir(parents=True,exist_ok=True)
if not (repo/'.git').exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/shreyammm/e-rate.git',str(repo)],check=True,timeout=240)
else:
    subprocess.run(['git','pull','--ff-only'],cwd=repo,check=True,timeout=240)
sys.path.insert(0,str(repo))
from scripts import kaggle_census as gpu
cc=gpu.c
rows=cc.load_corpus(repo/'data/fcc_corpus',None,'ocr_candidate_v1','augmented')
book=cc.load_book()
assert len(rows)==3231 and sum(bool(r['text'].strip()) for r in rows)==3230
out=Path('/kaggle/working/fcc_kaggle')
# Prior saved output is an immutable checkpoint input, even while this notebook stays private.
import kagglehub
try:
    prior=Path(kagglehub.notebook_output_download('shreyammm/fcc-e-rate-gpu-census',path=gpu.BUNDLE,force_download=True))
    print('Restored prior saved output:',gpu.restore(prior,out,rows,book),flush=True)
except Exception as exc:
    print('Prior output unavailable; local and GitHub GPU checkpoints will still be checked:',type(exc).__name__,flush=True)
gpus=subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],text=True)
print(gpus,flush=True)
assert gpus.count('Tesla T4')==2, 'This notebook is configured for the free T4 x2 environment'
if not shutil.which('zstd'):
    subprocess.run(['apt-get','update','-qq'],check=True,timeout=240)
    subprocess.run(['apt-get','install','-y','-qq','zstd'],check=True,timeout=120)
if not shutil.which('ollama'):
    subprocess.run('curl -fsSL https://ollama.com/install.sh | sh',shell=True,check=True,timeout=300)
env=dict(os.environ,OLLAMA_HOST='127.0.0.1:11434',OLLAMA_MODELS='/kaggle/temp/ollama-models',OLLAMA_CONTEXT_LENGTH='8192',OLLAMA_NUM_PARALLEL='1')
import urllib.request
try:
    urllib.request.urlopen('http://127.0.0.1:11434/api/tags',timeout=2)
except Exception:
    log=open('/kaggle/working/ollama.log','a')
    server=subprocess.Popen(['ollama','serve'],env=env,stdout=log,stderr=log)
    for _ in range(60):
        try:
            urllib.request.urlopen('http://127.0.0.1:11434/api/tags',timeout=2)
            break
        except Exception:
            time.sleep(1)
with open('/kaggle/working/model-download.log','w') as download_log:
    subprocess.run(['ollama','pull',gpu.MODEL],env=env,stdout=download_log,stderr=download_log,check=True,timeout=900)
print('Ready: local gpt-oss 20B, verified 3,230 readable filings.',flush=True)



In [ ]:
# Stops safely before Kaggle's 12-hour saved-session limit.
# Atomic checkpoints after each model window; compressed bundle and summary every ten filings.
# All remaining readable filings are attempted. Long appendices retain explicit prioritized-window flags.
summary=gpu.run(repo/'data/fcc_corpus',out,minutes=620,workers=1)
print('COMPLETED_READABLE_FILINGS',summary['completed_filings'],'OF',3230,flush=True)

